In [3]:
import numpy as np
import sys
import scipy

from scipy.integrate import solve_ivp

import matplotlib.pyplot as plt
from matplotlib import cm

# Neutron Diffusion in a Bare Spherical Fissile Assembly

**M6023 Advanced Quantitative Modelling: Assessment 3**

**Student ID:** 23360932014

This notebook develops a one-group neutron diffusion model for a bare spherical fissile assembly.

## Contents

1. Background


## 1. Background

### The physical question

A lump of fissile material, such as uranium-235 or plutonium-239, contains nuclei that can split when struck by a neutron, releasing energy and further neutrons. Each fission event releases on average $\nu \approx 2.4$ to $2.9$ neutrons, which may then go on to induce further fissions, be absorbed without causing fission, or leak out of the material entirely. Whether the neutron population grows, stays steady, or decays over time depends on the balance between these three fates, which in turn depends on the size, shape, and composition of the assembly.

The smallest mass of a given fissile material that can sustain a self-propagating chain reaction is called the **critical mass**. For a bare sphere, this is equivalent to asking for the **critical radius** $R_c$: below $R_c$ too many neutrons escape the surface and the reaction dies out; above $R_c$ the neutron population grows exponentially; at exactly $R_c$ the population is in steady state. The critical radius is not a free parameter of nature but a calculable quantity determined by the microscopic interaction probabilities of neutrons with the material, together with the geometry of the assembly.

This notebook develops a mathematical model of this balance, derives $R_c$ analytically, verifies the result numerically, and compares the critical radii of U-235 and Pu-239.

### Why this is a partial differential equation problem

The neutron flux $\varphi(\mathbf{r}, t)$ is a field: it takes a value at every point in space and every instant in time. Its evolution is governed by three competing processes which each act locally:

- **Production** from fission, proportional to the local flux
- **Absorption** by nuclei, also proportional to the local flux
- **Leakage** driven by spatial gradients in the flux, mathematically analogous to diffusion of heat or of a chemical species

Because leakage depends on spatial derivatives of $\varphi$ while production and absorption depend on $\varphi$ itself, the governing equation couples space and time through a partial differential equation of the form

$$\frac{\partial \varphi}{\partial t} = \underbrace{D \nabla^2 \varphi}_{\text{leakage}} + \underbrace{(\nu \Sigma_f - \Sigma_a)\varphi}_{\text{net production}}$$

The derivation of this equation from first principles, its specialisation to a spherical geometry, and its solution are the subject of the remainder of this notebook.

### Why the bare sphere, and why one-group theory

The bare sphere is the simplest geometry that captures the essential physics: it has a single length scale ($R$), it admits an analytical solution that can be used to verify numerical results, and it gives an upper bound on the critical radius for a given material, since any reflector placed around it would only reduce $R_c$. The bare-sphere critical radius is also historically significant, as it is the idealised calculation that underpinned the first fission device designs in the 1940s.

One-group diffusion theory treats all neutrons as having a single effective energy, collapsing what would otherwise be a seven-dimensional problem (three spatial, one temporal, one energy, two angular) to a two-dimensional one ($r$ and $t$) under spherical symmetry. The full picture requires either multi-group methods or the neutron transport equation, which are standard in research and industry but are substantially more involved than is warranted here. For a fast-spectrum bare metal assembly, where moderation is minimal and the neutron energy distribution is relatively narrow, one-group theory gives results within roughly 10 to 20 per cent of experimental critical masses, and the error sources are well-understood.

### What this notebook does

The remainder of the notebook:

1. States the modelling assumptions and their justifications (Section 2)
2. Derives the one-group neutron diffusion equation from a neutron balance on a control volume combined with Fick's law (Section 3)
3. Specialises to spherical symmetry and introduces the substitution $u = r\varphi$ to eliminate the coordinate singularity at the origin (Section 4)
4. Solves the steady-state problem analytically, giving $R_c = \pi \sqrt{D / (\nu \Sigma_f - \Sigma_a)}$ (Section 5)
5. Implements a finite-difference numerical scheme for the time-dependent equation (Section 6)
6. Verifies the numerical scheme against the analytical result (Section 7)
7. Demonstrates subcritical decay, critical steady state, and supercritical exponential growth (Section 8)
8. Compares U-235 and Pu-239 using standard cross-section data and discusses the physical origin of the difference (Section 9)
9. Discusses limitations and extensions (Section 10)